# 实验 20 — dbt source freshness

**目标**：让 dbt 监控上游数据 “新不新”。如果 source 数据停了（dlt pipeline 挂了 / 上游 API 挂了），下游 dbt 跑出来的还是“成功”但是基于过期数据 —— 这种沉默失败很可怕。

`dbt source freshness` 可以读 source 表的 loaded_at 字段，和当前时间比，发出 `warn_after` 或 `error_after` 警报。

dlt 给每个 raw 表自动写 `_dlt_load_id`，可以 join `_dlt_loads.inserted_at` 拿到加载时间——但 freshness 要求一个**列**做时间戳，最简单的做法是在 source 上配 `loaded_at_field: _dlt_load_id`（其格式是 ISO8601 timestamp 前缀，DuckDB 不直接接收，所以需要 hack）。

更可靠的做法是给 source 加一个 view 把 inserted_at 物化进来。下面演示这个做法。

In [ ]:
# 1) 给 raw_ecb.daily_rates 建一个带 loaded_at 时间戳的 view，让 dbt 用它做 freshness 检查
import duckdb
con = duckdb.connect('../data/sandbox.duckdb')
con.execute("""
    create or replace view raw_ecb.daily_rates_with_loaded_at as
    select dr.*, l.inserted_at as loaded_at
    from raw_ecb.daily_rates dr
    join raw_ecb._dlt_loads l on dr._dlt_load_id = l.load_id
""")
con.execute("select max(loaded_at) from raw_ecb.daily_rates_with_loaded_at").fetchall()

In [ ]:
# 2) 改 _sources.yml 加上 freshness 配置
from pathlib import Path
yml = Path('../dbt_project/models/staging/_sources.yml')
orig = yml.read_text()
yml.write_text('''version: 2

sources:
  - name: raw_ecb
    schema: raw_ecb
    description: "Raw ECB exchange rate data loaded by dlt from the Frankfurter API."
    tables:
      - name: daily_rates
        description: "Daily EUR-based exchange rates in long format."
        columns:
          - name: date
            tests: [not_null]
          - name: currency
            tests: [not_null]

      - name: daily_rates_with_loaded_at
        description: "daily_rates joined with _dlt_loads to expose loaded_at timestamp."
        loaded_at_field: loaded_at
        freshness:
          warn_after: {count: 7, period: day}
          error_after: {count: 30, period: day}

      - name: currencies_meta
        columns:
          - name: code
            tests: [not_null, unique]
''')
print(yml.read_text())

In [ ]:
# 3) 跑 dbt source freshness
import subprocess, os
r = subprocess.run(['uv','run','dbt','source','freshness'],
                   capture_output=True, text=True, cwd='../dbt_project',
                   env={**os.environ,'DBT_PROFILES_DIR':'.'})
print(r.stdout + r.stderr)

In [ ]:
# 4) 设个故意失败的阈值（warn_after=1 second）看效果
yml.write_text(yml.read_text().replace(
    'warn_after: {count: 7, period: day}',
    'warn_after: {count: 1, period: second}'
).replace(
    'error_after: {count: 30, period: day}',
    'error_after: {count: 10, period: second}'
))
r = subprocess.run(['uv','run','dbt','source','freshness'],
                   capture_output=True, text=True, cwd='../dbt_project',
                   env={**os.environ,'DBT_PROFILES_DIR':'.'})
print(r.stdout + r.stderr)

In [ ]:
# 5) freshness 结果也写进 target/sources.json，CI 里可以解析
import json
p = Path('../dbt_project/target/sources.json')
if p.exists():
    s = json.loads(p.read_text())
    for r in s.get('results', []):
        print(r.get('unique_id'), '->', r.get('status'), r.get('max_loaded_at'))

In [ ]:
# 还原 _sources.yml
yml.write_text(orig)

## 生产环境踩坑提示

- **freshness 是“最新行的时间”**，不是“跑成功的时间”。空表 / 部分有数据都可能蒙混过关；额外配 row-count 监控更全面。
- **filter**：可以加 `filter: loaded_at >= current_date - interval '7 days'` 排除老分区，只看最近窗口的新鲜度。
- **不只是查仓库**：在数据 lake / Kafka topic 那种没有“最新时间戳列”的源，要换其他工具（如 Monte Carlo / Great Expectations / Soda）。
- **freshness 失败不挡 build**：默认 `dbt build` 不会检查 freshness。需要单独跑 `dbt source freshness`，CI 里建议先跑这个再跑 build。
- **错配 timezone**：`loaded_at` 列要是 UTC，否则跟 `current_timestamp` 比对会偏 N 小时。

## 思考题

- 如何把 freshness 检查作为 `dbt build` 的前置？写一个 Makefile target `make build-fresh`。
- 如果上游 API 周末不出新数据（如汇率周末闭市），freshness 在周一就会假告警。怎么排除？（提示：`filter`）
- 不用 view 改写、不动 raw 表，能不能直接把 `_dlt_load_id` 当 freshness 字段？为什么 dbt 默认要 timestamp 列？